In [1]:
import json, csv

infile = "berlin_diplomatic.geojson"
outfile = "berlin_diplomatic.csv"

FIELDS = [
    "id","type","lat","lon",
    "name","office","diplomatic","consulate",
    "country","target",
    "addr:street","addr:housenumber","addr:postcode","addr:city",
    "website","contact:website","phone","contact:phone","email","contact:email"
]

def get_center(feat):
    g = feat.get("geometry") or {}
    t = g.get("type")
    c = g.get("coordinates")
    if t == "Point" and isinstance(c, list) and len(c) >= 2:
        return c[1], c[0]  # lat, lon
    # Overpass "out center" kann center in properties liefern:
    p = feat.get("properties") or {}
    if "center" in p and isinstance(p["center"], dict):
        return p["center"].get("lat"), p["center"].get("lon")
    return None, None

with open(infile, "r", encoding="utf-8") as f:
    gj = json.load(f)

features = gj.get("features", [])
with open(outfile, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    w.writeheader()
    for feat in features:
        p = feat.get("properties") or {}
        lat, lon = get_center(feat)
        row = {
            "id": p.get("@id") or p.get("id"),
            "type": p.get("type"),
            "lat": lat,
            "lon": lon,
        }
        for k in FIELDS:
            if k in row:
                continue
            row[k] = p.get(k)
        w.writerow(row)

print("Wrote", outfile, "rows:", len(features))


Wrote berlin_diplomatic.csv rows: 211
